# 00d. pandas IV: reshape, groupby, merge, linear algebra, and the course toolkit

The store keeps prices **long**; the pipeline works **wide**; universes, fundamentals and
rates arrive as separate tables to be joined **point-in-time**; and a portfolio's risk is
a matrix product. This notebook covers the moves between those forms, plus a map of every
library the course will use.

**How to work through this.** Before running a cell, say out loud what you expect it to
print, including the *shape* and the *labels*. Most pandas confusion is a wrong guess about
which labels come back, and you only notice if you guessed first. Exercises have an empty
cell beneath them and the expected answer above; match it, then move on.

## Setup

Synthetic prices from `portfolio_lab.data.synthetic`: the same eight tickers as week 1,
business days from 2015-01-01, no network and no `data/` needed, identical on every run
(`seed=7`). The generator returns the store's **long** format; `pivot` turns it into the
**wide** frame the course uses everywhere. Primer 00d explains `pivot`; for now take it on
trust.

The generator's drift is deliberately strong (so the momentum tests have something to
find), so price **levels** wander far from anything real: some tickers end at a fraction of
a cent on the dollar. Use these frames for mechanics, and don't read finance into the numbers.

In [1]:
import numpy as np
import pandas as pd

from portfolio_lab.data.returns import simple_returns
from portfolio_lab.data.synthetic import make_synthetic_prices

pd.set_option("display.precision", 4)
pd.set_option("display.width", 100)

TICKERS = ["SPY", "EFA", "IEF", "GLD", "AAPL", "MSFT", "JNJ", "XOM"]
long, truth = make_synthetic_prices(TICKERS, n_days=2520, seed=7)  # long format, like the store
long["date"] = pd.to_datetime(long["date"])
px = long.pivot(index="date", columns="ticker", values="close")    # wide: dates x tickers
px.columns.name = None
rets = simple_returns(px)                                           # your week-1 function
px.shape, rets.shape

((2520, 8), (2520, 8))

## 1. Long vs wide

| Long (tidy) | Wide |
|---|---|
| one row per `(date, ticker)` | one row per date, one column per ticker |
| how the store saves prices | what signals, covariance and the engine consume |
| easy to append, filter, join, group | easy to do per-date math across tickers |

`pivot` goes long to wide; `melt` or `stack` go wide to long.

In [2]:
long.head(3)

,date,ticker,close,volume
0,2015-01-01,SPY,100.7528,1000000.0
1,2015-01-01,EFA,100.5228,1000000.0
2,2015-01-01,IEF,100.6853,1000000.0


In [3]:
wide = long.pivot(index="date", columns="ticker", values="close")
back = wide.stack().rename("close").reset_index()   # wide -> long again
print(wide.shape, back.shape, len(long))
back.head(3)

(2520, 8) (20160, 3) 20160


,date,ticker,close
0,2015-01-01,AAPL,99.5859
1,2015-01-01,EFA,100.5228
2,2015-01-01,GLD,99.3121


`pivot` **refuses duplicate `(date, ticker)` keys**: it can't put two values in one cell.
That's one reason the store's `_validate_long_prices` rejects duplicates at write time.
`pivot_table` accepts duplicates by aggregating them (default: mean). Don't use it to paper
over bad data.

In [4]:
dup = pd.concat([long.head(2), long.head(1)])
try:
    dup.pivot(index="date", columns="ticker", values="close")
except ValueError as e:
    print("ValueError:", e)

ValueError: Index contains duplicate entries, cannot reshape


**MultiIndex.** `set_index(["date", "ticker"])` gives the long frame a two-level index.
`unstack()` moves the inner level into the columns (= pivot); `stack()` moves it back.
`.xs` selects one value of one level.

In [5]:
mi = long.set_index(["date", "ticker"])["close"]
print(mi.index.nlevels, mi.index.names)
print(mi.xs("SPY", level="ticker").head(2))
mi.unstack().iloc[:2, :3]

2 ['date', 'ticker']
date
2015-01-01    100.7528
2015-01-02     97.9358
Name: close, dtype: float64


ticker,AAPL,EFA,GLD
date,,,
2015-01-01,99.5859,100.5228,99.3121
2015-01-02,98.0027,99.7485,95.5645


**Exercise 1.** Convert `rets` to long format with columns `date`, `ticker`, `ret`,
dropping the NaN first row. Answer with its shape and first three rows. Then assert that
pivoting it back reproduces `rets.iloc[1:]` exactly (`pd.testing.assert_frame_equal`;
watch the column order and the `columns.name`).

Expected:

```
(20152, 3)

        date ticker     ret
0 2015-01-02   AAPL -0.0159
1 2015-01-02    EFA -0.0077
2 2015-01-02    GLD -0.0377
```

## 2. groupby: split, apply, combine

`groupby(key)` splits rows into groups, applies a function to each, and combines the
results. Two shapes of result:

- **`agg`** (or `.mean()`, `.last()`, ...) gives **one row per group**.
- **`transform`** gives **one value per original row**, the group's result broadcast
  back. Use it to demean within a group without losing the row alignment.

In [6]:
long.groupby("ticker")["close"].agg(["first", "last", "count"]).head(3)

,first,last,count
ticker,,,
AAPL,99.5859,157.5634,2520
EFA,100.5228,617.0917,2520
GLD,99.3121,0.7191,2520


In [7]:
lr = rets.iloc[1:].stack().rename("ret").rename_axis(["date", "ticker"]).reset_index()
lr["demeaned"] = lr["ret"] - lr.groupby("date")["ret"].transform("mean")
lr.head(3)

,date,ticker,ret,demeaned
0,2015-01-02,AAPL,-0.0159,-0.0009
1,2015-01-02,EFA,-0.0077,0.0073
2,2015-01-02,GLD,-0.0377,-0.0228


That per-date `transform` is the long-format twin of `rets.sub(rets.mean(axis=1), axis=0)`
from 00b. Both are right; use whichever form the data is already in.

**Grouping by time.** Pass an array computed from the index:

In [8]:
rets.groupby(rets.index.year)[["SPY", "IEF"]].std().mul(np.sqrt(252)).head(3)

,SPY,IEF
date,,
2015,0.2158,0.2016
2016,0.2078,0.1949
2017,0.2341,0.2180


**Grouping tickers into sleeves.** The universe CSV assigns each ticker a sleeve. On a
wide frame, transpose so tickers become rows, group the rows, and transpose back
(pandas 3 removed `groupby(axis=1)`):

In [9]:
from portfolio_lab.data.store import REPO_ROOT
from portfolio_lab.data.universe import Universe

uni = Universe.from_csv(str(REPO_ROOT / "configs" / "universe_week1.csv"))
sleeves = pd.Series(uni.sleeves)          # ticker -> sleeve
print(sleeves.to_dict())
sleeve_rets = rets.T.groupby(sleeves).mean().T   # equal-weight return per sleeve per day
sleeve_rets.dropna().head(3)

{'SPY': 'equity', 'EFA': 'equity', 'IEF': 'duration', 'GLD': 'real_asset', 'AAPL': 'equity', 'MSFT': 'equity', 'JNJ': 'equity', 'XOM': 'equity'}


,duration,equity,real_asset
date,,,
2015-01-02,-0.0086,-0.0122,-0.0377
2015-01-05,-0.0025,-0.0006,0.0008
2015-01-06,-0.0128,-0.0121,-0.0307


**Exercise 2.** Annualised volatility of each ticker, then the **average** of those vols
within each sleeve. (Not the vol of the sleeve's average return: those differ, and
the difference is diversification, week 3's topic.) Answer with a Series indexed by
sleeve, rounded to 4 places.

Expected:

```
duration      0.2079
equity        0.1883
real_asset    0.2540
dtype: float64
```

## 3. Combining tables

| Tool | Joins on | Use |
|---|---|---|
| `pd.concat([a, b])` | stacks rows (`axis=0`) or columns (`axis=1`), aligning the other axis | append snapshots; put series side by side |
| `a.join(b)` | the index | add columns from a frame with the same dates |
| `a.merge(b, on=..., how=...)` | columns, like SQL `JOIN` | attach attributes (sleeve, sector) to long rows |
| `pd.merge_asof(a, b, on=..., by=...)` | the **latest `b` row at or before** each `a` row | point-in-time joins |

Pass `validate="many_to_one"` (or `"one_to_one"`) to `merge`. It raises if the key isn't
unique on the side you claimed, which catches the silent row explosion that duplicate
keys cause.

In [10]:
attrs = sleeves.rename("sleeve").rename_axis("ticker").reset_index()
tagged = lr.merge(attrs, on="ticker", how="left", validate="many_to_one")
tagged.head(3)

,date,ticker,ret,demeaned,sleeve
0,2015-01-02,AAPL,-0.0159,-0.0009,equity
1,2015-01-02,EFA,-0.0077,0.0073,equity
2,2015-01-02,GLD,-0.0377,-0.0228,real_asset


**`merge_asof`: the point-in-time join.** Company fundamentals are *about* a quarter but
*published* weeks later. A backtest may only use a number once it has been published. In
week 6 you'll build a value signal with exactly this lag. Here's a toy version:

In [11]:
reports = pd.DataFrame({
    "ticker":   ["AAPL", "AAPL", "MSFT", "MSFT"],
    "period":   pd.to_datetime(["2019-12-31", "2020-03-31", "2019-12-31", "2020-03-31"]),
    "published": pd.to_datetime(["2020-01-28", "2020-04-30", "2020-01-29", "2020-04-29"]),
    "book_value": [90.0, 78.0, 102.0, 110.0],
})
month_ends = pd.DataFrame({"date": pd.to_datetime(["2020-01-31", "2020-03-31", "2020-04-30"])})
grid = month_ends.merge(pd.DataFrame({"ticker": ["AAPL", "MSFT"]}), how="cross")

pit = pd.merge_asof(
    grid.sort_values("date"), reports.sort_values("published"),
    left_on="date", right_on="published", by="ticker", direction="backward",
)
pit[["date", "ticker", "period", "published", "book_value"]]

,date,ticker,period,published,book_value
0,2020-01-31,AAPL,2019-12-31,2020-01-28,90.0
1,2020-01-31,MSFT,2019-12-31,2020-01-29,102.0
2,2020-03-31,AAPL,2019-12-31,2020-01-28,90.0
3,2020-03-31,MSFT,2019-12-31,2020-01-29,102.0
4,2020-04-30,AAPL,2020-03-31,2020-04-30,78.0
5,2020-04-30,MSFT,2020-03-31,2020-04-29,110.0


Read the 2020-03-31 rows. Q1's numbers exist on the company's books that day, but they
aren't *published* yet, so the join still returns the December report. Joining on `period`
instead of `published` is the lookahead bug; it quietly gives you the March figure a
month early. Both frames must be **sorted** on their join key, or `merge_asof` raises.

Now look at AAPL on 2020-04-30: the report was published *that day*. Companies usually
report after the close, so a signal computed at that close couldn't really have seen it.
That's why real pipelines add a lag on top of the publication date (Exercise 3).

**Exercise 3.** Repeat the join above, but assume you only learn of a report **5
business days after** `published` (the data vendor's lag). Add a column
`usable = published + pd.offsets.BDay(5)` and join on it instead. Answer with the rows
for 2020-04-30. Both tickers now see the December report. For each ticker, what is the
**smallest** lag in business days that would have changed its answer, and why do they
differ? (Look at `published`.)

Expected:

```
        date ticker     period     usable  book_value
4 2020-04-30   AAPL 2019-12-31 2020-02-04        90.0
5 2020-04-30   MSFT 2019-12-31 2020-02-05       102.0
```

**Membership windows** (week 5): the universe says a ticker is a member from
`member_from` until `member_to`. Keeping only rows where the date falls inside the window
is a merge followed by a mask:

In [12]:
members = pd.DataFrame({
    "ticker": ["AAPL", "XOM"],
    "member_from": pd.to_datetime(["2015-01-01", "2015-01-01"]),
    "member_to": pd.to_datetime([pd.NaT, "2018-06-30"]),   # XOM "leaves" mid-2018
})
m = lr.merge(members, on="ticker", how="inner")
inside = (m["date"] >= m["member_from"]) & (m["member_to"].isna() | (m["date"] <= m["member_to"]))
m[inside].groupby("ticker")["date"].agg(["min", "max", "count"])

,min,max,count
ticker,,,
AAPL,2015-01-02,2024-08-28,2519
XOM,2015-01-02,2018-06-29,911


## 4. Linear algebra: from frames to portfolio risk

The covariance matrix is a DataFrame with tickers on both axes. The course convention is
**annualised** (x 252) at the estimator boundary.

In [13]:
cov = rets.cov() * 252
corr = rets.corr()
cov.iloc[:3, :3]

,AAPL,EFA,GLD
AAPL,0.0290,0.0164,0.0071
EFA,0.0164,0.0457,0.0191
GLD,0.0071,0.0191,0.0645


Portfolio variance is $w^\top \Sigma w$. With labelled objects, `@` **aligns labels**
and raises if they don't match, which is exactly the safety you want when weights
come from one module and covariance from another:

In [14]:
w = pd.Series(1 / len(TICKERS), index=TICKERS)   # equal weight, in TICKERS order
var_p = w @ cov @ w                             # cov's columns are alphabetical; @ aligns
print(f"equal-weight vol from the covariance: {np.sqrt(var_p):.4%}")

try:
    pd.Series([0.5, 0.5], index=["SPY", "BTC"]) @ cov
except ValueError as e:
    print("ValueError:", e)

equal-weight vol from the covariance: 13.3209%
ValueError: matrices are not aligned


`.to_numpy()` drops the labels for speed, for example inside an optimiser, and then
**you** are responsible for the order. Put labels back straight after:
`pd.Series(result, index=cov.columns)`. Most mix-ups between tickers happen at that step.

Correlation from covariance: $\rho = D^{-1} \Sigma D^{-1}$, with $D$ the diagonal of
vols. `np.outer` builds the matrix of $\sigma_i \sigma_j$ products:

In [15]:
vols = np.sqrt(np.diag(cov))
rho = cov / np.outer(vols, vols)
print(np.allclose(rho, corr))

True


**Exercise 4.** Equal-weight portfolio vol **two ways**, and show they agree to 10 decimal
places: (a) `sqrt(w @ cov @ w)`; (b) build the daily portfolio return series
`rets @ w` (a matrix times a vector) and take its annualised std. Answer with the value
rounded to 6 places. Then think: why do (a) and (b) agree only because the weights are
reset every day here, and why would a real buy-and-hold portfolio differ? (Weight drift:
`CLAUDE.md`'s second invariant.)

Expected:

```
0.133209
```

## 5. SQL and LINQ to pandas

You know SQL and LINQ from dashboard work. Here's the mapping:

| SQL | LINQ (C#) | pandas |
|---|---|---|
| `SELECT a, b` | `.Select(x => new {x.A, x.B})` | `df[["a", "b"]]` |
| `WHERE a > 0 AND b = 'x'` | `.Where(...)` | `df[(df.a > 0) & (df.b == "x")]` or `df.query("a > 0 and b == 'x'")` |
| `ORDER BY a DESC` | `.OrderByDescending(x => x.A)` | `df.sort_values("a", ascending=False)` |
| `GROUP BY k` + `AVG(v)` | `.GroupBy(x => x.K).Select(g => g.Average(...))` | `df.groupby("k")["v"].mean()` |
| `JOIN ... ON` | `.Join(...)` | `df.merge(other, on=..., how=...)` |
| `UNION ALL` | `.Concat(...)` | `pd.concat([a, b])` |
| `LAG(v) OVER (PARTITION BY k ORDER BY d)` | (no direct equivalent) | `df.sort_values("d").groupby("k")["v"].shift(1)` |
| `AVG(v) OVER (... ROWS 62 PRECEDING)` | (no direct equivalent) | `s.rolling(63).mean()` |
| `LIMIT 5` | `.Take(5)` | `df.head(5)` |
| `CASE WHEN` | ternary in `Select` | `np.where(cond, a, b)` |

The big difference is **eager vs lazy**. LINQ and SQL build a query and run it once;
pandas runs every line immediately and holds the result in memory. On the S&P 500 over
20 years (week 5, about 10 million rows) that's still fine.

**DuckDB runs SQL straight on DataFrames.** It's already a dependency: the store uses it
to read parquet snapshots. It's handy for exploring in a notebook, and a good way to
check your pandas against SQL you already trust. (In the package, only `store.py` may
import it; that's a `CLAUDE.md` rule.)

In [16]:
import duckdb

duckdb.sql("""
    SELECT ticker, COUNT(*) AS n, ROUND(STDDEV_SAMP(ret) * SQRT(252), 4) AS ann_vol
    FROM lr GROUP BY ticker ORDER BY ann_vol DESC LIMIT 3
""").df()

,ticker,n,ann_vol
0,GLD,2519,0.2540
1,SPY,2519,0.2211
2,EFA,2519,0.2138


## 6. The course toolkit

Everything the syllabus will ask for, and where you'll first need it. Versions are what
`.venv` has today.

In [17]:
import importlib
for name in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "duckdb", "pyarrow",
             "statsmodels"]:
    try:
        print(f"{name:12s} {importlib.import_module(name).__version__}")
    except ImportError:
        print(f"{name:12s} not installed")

numpy        2.5.3
pandas       3.0.5
scipy        1.18.1


sklearn      1.9.1
matplotlib   3.11.2
duckdb       1.5.5
pyarrow      25.0.1
statsmodels  not installed


| Library | For | First needed | pandas skills it leans on |
|---|---|---|---|
| **pandas** | everything tabular | week 1 | all four primers |
| **numpy** | arrays, `@`, `np.where`, random draws for bootstraps | week 1 | `.to_numpy()` and putting labels back |
| **DuckDB + pyarrow** | the snapshot store (parquet); SQL exploration | week 1 | long format, `pivot` |
| **matplotlib** | the tearsheet; frontier and drawdown plots | week 3 | `df.plot()` wraps it |
| **scipy** (`optimize`, `stats`) | constrained mean-variance, normal quantiles for VaR, Black-Scholes | weeks 3, 4, 10 | Series to array and back |
| **scikit-learn** | `LedoitWolf` to check your hand-derived shrinkage; purged CV in week 7 | weeks 4, 7 | wide returns in, labelled matrix out |
| **statsmodels** (not installed yet) | Fama-French factor regressions with standard errors | week 6 | `merge` returns with factor data on date |

Week by week, the pandas moves you'll lean on:

| Week | Moves |
|---|---|
| 1 data integrity | `pivot`, `shift`, `reindex` + `ffill`, `as_of` snapshots, `agg` |
| 2 engine | numpy-array loop over dates, `.loc` writes, weight drift, `cumprod` |
| 3 mean-variance | `cov`, `@`, `to_numpy` round trips, resampling rows for the bootstrap |
| 4 risk | `rolling`, `ewm`, `quantile` for VaR, planted covariance vs estimated |
| 5 survivorship | long format, `merge` on membership windows, `groupby`, DuckDB at scale |
| 6 signals, factors | `merge_asof` with reporting lag, `rank(axis=1)`, `merge` with Fama-French |
| 7 overfitting | a trial ledger (a DataFrame you append to), `groupby` over trials |
| 8-10 instruments | aligning rates and futures calendars, `concat` for roll-adjusted contracts |
| 11-12 attribution, capstone | `groupby` by sleeve and period, pivot tables, `plot` |

## 7. Reading: where this primer's content comes from

Most university data-science syllabi teach pandas from the same few texts:

- **McKinney, *Python for Data Analysis*, 3rd ed.** (O'Reilly, 2022; updated for pandas
  2.0). Free online: https://wesmckinney.com/book/. Ch. 5 basics, 6 loading and
  storage, 7 cleaning, 8 join/combine/reshape, 10 groupby, 11 time series. If you read
  one book, read this one.
- **UC Berkeley Data 100**, *Principles and Techniques of Data Science*
  (https://ds100.org/), and its textbook: **Lau, Gonzalez & Nolan, *Learning Data
  Science*** (O'Reilly, 2023), free at https://learningds.org/. Ch. 6-7 teach dataframes
  in pandas and SQL side by side, which suits your background.
- **VanderPlas, *Python Data Science Handbook*** ch. 3 "Data Manipulation with Pandas":
  https://jakevdp.github.io/PythonDataScienceHandbook/. Covers hierarchical indexing,
  aggregation and grouping, pivot tables and time series.
- **QuantEcon, *Python Programming for Economics and Finance***, lectures "Pandas" and
  "Pandas for Panel Data": https://python-programming.quantecon.org/pandas.html. pandas
  taught on financial and economic data; the closest in spirit to this course.
- **pandas documentation** (3.x): the user guide
  (https://pandas.pydata.org/docs/user_guide/index.html), especially "Group by:
  split-apply-combine", "Merge, join, concatenate and compare", "Reshaping and pivot
  tables"; and "Comparison with SQL"
  (https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html).
- **pandas 3.0 release notes** (https://pandas.pydata.org/docs/whatsnew/v3.0.0.html),
  because most tutorials predate Copy-on-Write and the `str` dtype.